# 06 · Comparar el tamaño de chunk con nomic-embed-text en Qdrant

Medimos cómo cambia la recuperación según el troceo del corpus de la documentación de Python 3.15. Se fija el modelo (`nomic-embed-text`, Ollama), la base (Qdrant) y k=4; solo cambian el tamaño y el solape del chunk. Se evalúa con el golden set de `docs/GOLDEN_SET_PYTHON_DOCS.md` y las evidencias de `docs/golden_set_evidencias.json`.

Necesita Qdrant en `localhost:6333` (`docker compose up -d`) y Ollama con `nomic-embed-text` (`ollama pull nomic-embed-text`).

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

`CONFIGURACIONES` es el único sitio donde se declaran las configuraciones `(tamano, solape)`; el resto de celdas las recorre.

In [ ]:
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, evidencias_fuera_de_chunks, fuentes_ausentes,
    leer_evidencias, leer_golden_set, preparar_documento, preparar_query,
    rango_primer_acierto, rango_primer_acierto_evidencia, trocear_corpus,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CSV = RAIZ / "data" / "resultados" / "tamano_chunk.csv"
MODELO = "nomic"
K = 4
LOTE_EMBEDDING = 64   # chunks por llamada al modelo
LOTE_INSERCION = 500  # puntos por inserción
# (tamano, solape) en caracteres
CONFIGURACIONES = [(300, 0), (500, 0), (1000, 0), (1500, 0), (500, 100), (1000, 200)]

## 3. Conexión con Qdrant y Ollama

Se comprueba antes de calcular nada, para fallar pronto si falta un servicio. Si Qdrant u Ollama no responden, la celda para con un mensaje que dice qué hacer.

In [ ]:
qdrant = QdrantClient("localhost", port=6333)
try:
    qdrant.get_collections()
except Exception as error:
    raise RuntimeError(
        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d` y vuelve a ejecutar."
    ) from error

modelo_ollama = MODELOS[MODELO]["nombre"]
try:
    ollama.show(modelo_ollama)
except ConnectionError as error:
    raise RuntimeError("Ollama no responde. Arráncalo (`ollama serve` o la app) y vuelve a ejecutar.") from error
except ollama.ResponseError as error:
    raise RuntimeError(
        f"Ollama no tiene el modelo {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}` y vuelve a ejecutar."
    ) from error

print("Qdrant:", [col.name for col in qdrant.get_collections().collections])
print("Ollama:", modelo_ollama, "disponible")

## 4. Preguntas y evidencias del golden set

Se leen de las tablas de `docs/GOLDEN_SET_PYTHON_DOCS.md`; ninguna pregunta está escrita en el código. Antes de evaluar se avisa, con el ID y la ruta, de las fuentes que no existen en el corpus. La evidencia literal de cada pregunta A-D sale de `docs/golden_set_evidencias.json`.

In [ ]:
preguntas = leer_golden_set(RUTA_GOLDEN)
ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)
for id_, ruta in ausentes:
    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")
print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())

evidencias = leer_evidencias(RUTA_EVIDENCIAS)

## 5. Troceo por configuración

Para cada `(tamano, solape)` se trocea el corpus y se cuentan los chunks. Con `evidencias_fuera_de_chunks` se listan las evidencias que el troceo deja partidas (pares ID y fuente); no paran la ejecución, pero esa pregunta no podrá acertar por evidencia.

In [ ]:
def nombre_coleccion(tamano: int, solape: int) -> str:
    return f"pydocs_{MODELO}_c{tamano}_s{solape}"


chunks_por_config: dict[tuple[int, int], list[dict]] = {}
partidas: dict[tuple[int, int], list[tuple[str, str]]] = {}
for tamano, solape in CONFIGURACIONES:
    chunks = trocear_corpus(CARPETA_CORPUS, tamano, solape)
    chunks_por_config[(tamano, solape)] = chunks
    partidas[(tamano, solape)] = evidencias_fuera_de_chunks(evidencias, chunks)
    print(f"tamaño {tamano}, solape {solape}: {len(chunks)} chunks, {len(partidas[(tamano, solape)])} evidencias partidas")
    for id_, fuente in partidas[(tamano, solape)]:
        print(f"    partida: {id_} en {fuente}")

## 6. Embeddings e indexación por configuración

Una colección por configuración (`pydocs_nomic_c{tamano}_s{solape}`), con distancia coseno. Si ya existe, se borra y se vuelve a crear. Los prefijos salen de `MODELOS`. Los vectores se calculan una vez por configuración y no se guardan. El tiempo de indexación es embeddings más inserción.

In [ ]:
def embeber(textos: list[str], barra: bool = False, descripcion: str = "") -> list[list[float]]:
    """Embeddings de una lista de textos ya preparados, en lotes fijos."""
    vectores: list[list[float]] = []
    for i in tqdm(range(0, len(textos), LOTE_EMBEDDING), desc=descripcion, disable=not barra):
        lote = textos[i:i + LOTE_EMBEDDING]
        vectores.extend(ollama.embed(model=modelo_ollama, input=lote).embeddings)
    return vectores


t_indexacion: dict[tuple[int, int], float] = {}
for tamano, solape in CONFIGURACIONES:
    chunks = chunks_por_config[(tamano, solape)]
    nombre = nombre_coleccion(tamano, solape)
    inicio = time.perf_counter()
    vectores = embeber(
        [preparar_documento(MODELO, c["texto"]) for c in chunks], barra=True, descripcion=nombre
    )
    if qdrant.collection_exists(nombre):
        qdrant.delete_collection(nombre)
    qdrant.create_collection(
        nombre, vectors_config=VectorParams(size=len(vectores[0]), distance=Distance.COSINE)
    )
    for i in range(0, len(chunks), LOTE_INSERCION):
        qdrant.upsert(nombre, points=[
            PointStruct(id=c["id"], vector=v, payload={"fuente": c["fuente"], "texto": c["texto"]})
            for c, v in zip(chunks[i:i + LOTE_INSERCION], vectores[i:i + LOTE_INSERCION])
        ])
    t_indexacion[(tamano, solape)] = time.perf_counter() - inicio
    print(f"{nombre}: {len(chunks)} chunks en {t_indexacion[(tamano, solape)]:.1f} s")

## 7. Evaluación con k=4

Para cada configuración y cada pregunta se recuperan 4 resultados y se mide la latencia (embedding de la query más búsqueda). Se calculan dos rangos: `rango` (acierta si la `fuente` del chunk es una de las fuentes de la pregunta) y `rango_evidencia` (además, el texto del chunk contiene la evidencia de esa fuente). Se guarda el score coseno del mejor resultado; en las preguntas E (sin fuentes) indica cuánta relevancia «inventa» el modelo.

In [ ]:
resultados = []
for tamano, solape in CONFIGURACIONES:
    nombre = nombre_coleccion(tamano, solape)
    for p in tqdm(preguntas, desc=nombre):
        inicio = time.perf_counter()
        vector = embeber([preparar_query(MODELO, p["query"])])[0]
        res = qdrant.query_points(nombre, query=vector, limit=K, with_payload=True).points
        latencia_ms = (time.perf_counter() - inicio) * 1000
        recuperados = [{"fuente": r.payload["fuente"], "texto": r.payload["texto"]} for r in res]
        resultados.append({
            "tamano": tamano, "solape": solape, "id": p["id"], "categoria": p["categoria"],
            "rango": rango_primer_acierto([r["fuente"] for r in recuperados], p["fuentes"]) if p["fuentes"] else None,
            "rango_evidencia": rango_primer_acierto_evidencia(recuperados, evidencias[p["id"]]) if p["fuentes"] else None,
            "con_fuentes": bool(p["fuentes"]),
            "score": res[0].score, "latencia_ms": latencia_ms,
        })

df = pd.DataFrame(resultados)
print(len(df), "evaluaciones")

## 8. Métricas por configuración (preguntas A-D)

hit@1, hit@4 y MRR@4 sobre las preguntas con fuentes, con los dos criterios: por fuente y por evidencia (más estricto, prefijo `ev_`).

In [ ]:
def metricas(grupo: pd.DataFrame) -> pd.Series:
    """Métricas por fuente y por evidencia (prefijo `ev_`)."""
    filas = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        filas.update({prefijo + n: v for n, v in calcular_metricas(rangos, K).items()})
    return pd.Series(filas)


con_fuentes = df[df["con_fuentes"]]
por_config = con_fuentes.groupby(["tamano", "solape"]).apply(metricas, include_groups=False)
por_config

## 9. Desglose por categoría (A-D)

In [ ]:
por_categoria = con_fuentes.groupby(["categoria", "tamano", "solape"]).apply(metricas, include_groups=False)
por_categoria

## 10. Preguntas E: score del mejor resultado

Las preguntas E no tienen respuesta en el corpus. Un score alto en el mejor resultado indica que el modelo «inventa» relevancia.

In [ ]:
df[~df["con_fuentes"]].pivot_table(index="id", columns=["tamano", "solape"], values="score").round(3)

## 11. Tabla resumen y CSV

Una fila por configuración. El tiempo de indexación es embeddings más inserción; el score E es la media del mejor score en las preguntas E. Se guarda en `data/resultados/tamano_chunk.csv` (la carpeta se crea si no existe).

In [ ]:
filas = []
for tamano, solape in CONFIGURACIONES:
    m = por_config.loc[(tamano, solape)]
    sub = df[(df["tamano"] == tamano) & (df["solape"] == solape)]
    filas.append({
        "tamano": tamano, "solape": solape,
        "chunks": len(chunks_por_config[(tamano, solape)]),
        "hit@1": m["hit@1"], f"hit@{K}": m[f"hit@{K}"], "MRR": m["mrr"],
        "ev_hit@1": m["ev_hit@1"], f"ev_hit@{K}": m[f"ev_hit@{K}"], "ev_MRR": m["ev_mrr"],
        "latencia media (ms)": sub["latencia_ms"].mean(),
        "indexación (s)": t_indexacion[(tamano, solape)],
        "score medio E": sub[~sub["con_fuentes"]]["score"].mean(),
        "evidencias partidas": len(partidas[(tamano, solape)]),
    })
resumen = pd.DataFrame(filas)
RUTA_CSV.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CSV, index=False)
print(f"Guardado en {RUTA_CSV.relative_to(RAIZ)}")
resumen.round(3)